# Reinforcement Learning: MDPs to Policy Gradients
**Level:** Professional | **Interactive Machine Learning Curriculum**

> Markov Decision Processes, Bellman Equations, Q-Learning, Deep Q-Networks (DQN), REINFORCE, and Proximal Policy Optimization (PPO).

---

### 📖 Conceptual Foundations

#### The Reinforcement Learning Problem

Unlike Supervised Learning (which learns from explicit target labels) and Unsupervised Learning (which discovers hidden patterns in unlabeled data), **Reinforcement Learning (RL)** is an agent-centric paradigm. An **Agent** interacts with an external **Environment** through trial and error over discrete time steps $t$, taking **Actions** ($a_t \in \mathcal{A}$), observing new **States** ($s_{t+1} \in \mathcal{S}$), and receiving scalar **Rewards** ($r_t \in \mathbb{R}$) with the goal of maximizing expected cumulative discounted return $G_t = \sum_{k=0}^\infty \gamma^k r_{t+k}$.


#### Markov Decision Process (MDP) & Bellman Optimality Equations

An MDP is a 5-tuple $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$ where the **Markov Property** holds: $P(s_{t+1} \mid s_t, a_t, \dots, s_0, a_0) = P(s_{t+1} \mid s_t, a_t)$.

1. **Bellman Optimality Equation for State-Value $V^*(s)$:**
$$V^*(s) = \max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V^*(s') \right]$$

2. **Bellman Optimality Equation for Action-Value $Q^*(s, a)$:**
$$Q^*(s, a) = \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) \max_{a' \in \mathcal{A}} Q^*(s', a')$$


#### Reinforcement Learning Taxonomy

| Algorithm Family | Core Mechanism | Policy Type | Strengths | Weaknesses |
| --- | --- | --- | --- | --- |
| Value-Based (Q-Learning / DQN) | Approximates action-value function $Q(s, a)$; selects greedy action $\arg\max_a Q(s, a)$. | Off-Policy (learns target policy from exploratory behavior). | Sample efficient; handles discrete action spaces well. | Cannot natively handle continuous action spaces; prone to value overestimation. |
| Policy-Based (REINFORCE) | Directly parameterizes policy $\pi_\theta(a \mid s)$ and updates weights via gradient ascent on expected return. | On-Policy. | Naturally handles continuous actions; can learn stochastic policies. | High variance in gradient estimates; slow sample efficiency. |
| Actor-Critic (A2C / PPO) | Actor parameterizes policy $\pi_\theta(a \mid s)$; Critic parameterizes value function $V_\phi(s)$ to compute advantage $A(s, a)$. | PPO is On-Policy (with importance sampling surrogate). | State-of-the-art stability; industry standard for robotics and LLM alignment (RLHF). | Requires tuning multiple loss components and value networks. |


#### The Deep Q-Network (DQN) Breakthrough (Mnih et al., DeepMind 2015)

Directly combining Q-learning with non-linear neural networks is notoriously unstable ('The Deadly Triad'). DQN solved this with two innovations:
1. **Experience Replay Buffer:** Stores transitions $(s, a, r, s', d)$ and samples random uncorrelated mini-batches, breaking temporal correlations.
2. **Target Network ($\hat{Q}$):** Uses a separate, slowly updated set of weights $\theta^-$ to compute target values $r + \gamma \max_{a'} Q(s', a'; \theta^-)$, preventing feedback loop instability.


#### Proximal Policy Optimization (PPO, Schulman et al., 2017)

Standard policy gradient steps can cause catastrophic policy collapses if step sizes are too large. **PPO** enforces stability by clipping the probability ratio $r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{old}}(a_t \mid s_t)}$ within $[1 - \epsilon, 1 + \epsilon]$ (typically $\epsilon = 0.2$):

$$L^{\text{CLIP}}(\theta) = \hat{\mathbb{E}}_t \left[ \min\left( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$

This prevents the new policy from deviating excessively from the old policy, providing smooth, monotonic improvements.



In [ ]:
# %matplotlib inline
import numpy as np

# Simple 4-state 1D Line World: [State 0, State 1, State 2, Goal (State 3)]
# Actions: 0 = Left, 1 = Right
num_states = 4
num_actions = 2
Q = np.zeros((num_states, num_actions))

lr = 0.1
gamma = 0.95
epsilon = 0.2

def step(state, action):
    if action == 0: # Left
        next_state = max(0, state - 1)
    else: # Right
        next_state = min(num_states - 1, state + 1)
    reward = 10.0 if next_state == 3 else -0.1
    done = (next_state == 3)
    return next_state, reward, done

# Train 200 episodes
np.random.seed(42)
for ep in range(200):
    s = 0
    done = False
    while not done:
        # Epsilon-greedy action selection
        if np.random.rand() < epsilon:
            a = np.random.randint(num_actions)
        else:
            a = np.argmax(Q[s, :])
            
        s_next, r, done = step(s, a)
        
        # Bellman update equation: Q(s, a) <- Q(s, a) + lr * (r + gamma * max_a' Q(s', a') - Q(s, a))
        td_target = r + (0 if done else gamma * np.max(Q[s_next, :]))
        td_error = td_target - Q[s, a]
        Q[s, a] += lr * td_error
        s = s_next

print('Learned Optimal Q-Table [Rows = States 0..3, Cols = [Left, Right]]:')
print(np.round(Q, 2))
action_names = ['Left', 'Right']
for s in range(3):
    best_a = action_names[np.argmax(Q[s, :])]
    print(f'State {s}: Best Action -> {best_a} (Q={np.max(Q[s, :]):.2f})')


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

